# 06a validate

Step 6 of the guide has two halves. This notebook is the validator's half: it trains nothing and reads the test scores that steps 4 and 5 saved. The ablations that need new training runs are notebook 06b.

| check | question it answers | guide item | talk it tests |
|---|---|---|---|
| paired bootstrap | is the transformer's lead over XGBoost larger than the noise? | 95% CI on the difference | WF: transformer beats the feature model |
| monthly stability | does recall at a fixed cutoff hold over the six test months? | time stability | Wealthsimple: drift monitoring |
| PSI per token field | did the input distribution move between the train and test periods? | PSI, train vs test | Wealthsimple: input data quality |
| subgroups | where is each model weak: category, young cards, cardholder groups? | per-group recall | WF: long history is the driver, so short history is the risk |
| error analysis | what does each model catch that the other misses? | 20 + 20 with decode() | what the sequence model actually learned |
| ceiling | are all scores near perfect because Sparkov's fraud is rule-generated? | synthetic-data caveat | guide: read the fixed-alert-rate gaps, not the levels |

Both models are compared at the same alert rate (0.6% of test rows), with one cutoff fixed on the whole test set, the way a deployed threshold is. Seed 0 of each model is the subject of the row-level checks; the bootstrap and the seed-pair table show how much seed noise there is.

In [ ]:
import os, sys, glob, shutil
try:
    from google.colab import drive, userdata                      # Colab: mount Drive, pull the repo, sync new notebooks to Drive
    drive.mount("/content/drive")
    tok = userdata.get("GH_TOKEN")
    if not os.path.exists("/content/transaction-fm"):
        !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
    else:
        !cd /content/transaction-fm && git pull -q
    %pip install -q datasets xgboost
    %cd /content/transaction-fm
except ImportError:                                               # laptop: this notebook is the repo copy; data under $TFM_ROOT
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, "src", "tfm")):
        assert here != os.path.dirname(here), "run this notebook from inside the transaction-fm repo"
        here = os.path.dirname(here)
    %cd {here}
sys.path.insert(0, os.path.abspath("src"))
from tfm.runtime import root, describe, on_colab
ROOT = root()
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
if on_colab():
    for f in glob.glob("notebooks/*.ipynb"):                      # new notebooks from the repo appear in Drive
        dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok |", describe(), "| ROOT =", ROOT)

## Stage 6a-1: load the test rows and the saved scores

In [ ]:
import json, time, os, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from tfm.tokenizer import TransactionTokenizer
from tfm.baseline import build_features
from tfm.metrics import evaluate_scores
from tfm.validate import (bootstrap_diff, monthly_metrics, psi_by_field, subgroup_metrics, disagreements,
                          history_length, history_bucket, global_cutoff)

P, PRE, RES = f"{ROOT}/data/processed", f"{ROOT}/pretrain/best", f"{ROOT}/results"
os.makedirs(RES, exist_ok=True)
df = pd.read_parquet(f"{P}/transactions.parquet")
cards = pd.read_parquet(f"{P}/cards.parquet")
tok = TransactionTokenizer.load(f"{PRE}/tokenizer.json")
ids_all = tok.encode(df)
FIELDS = ["TX"] + (["DT"] if tok.config["use_dt"] else []) + ["HOUR", "CAT", "MER", "AMT"] + (["DIST"] if tok.config["use_dist"] else [])
card, split, fraud = df["card_id"].to_numpy(), df["split"].to_numpy(), df["is_fraud"].to_numpy()
te = split == "test"; yt, ct = fraud[te], card[te]; test = df[te].reset_index(drop=True)
ALERT = 0.006

MODELS = {"transformer": "finetune/pretrained_H12/seed{k}", "transformer scratch": "finetune/scratch_H12/seed{k}",
          "xgboost": "baseline/full/seed{k}", "xgboost current-only": "baseline/current/seed{k}"}
scores = {}
for name, pat in MODELS.items():
    scores[name] = []
    for k in (0, 1, 2):
        path = f"{ROOT}/{pat.format(k=k)}/test_scores.npy"
        if os.path.exists(path):
            p = np.load(path); assert len(p) == len(yt), (name, k, len(p), len(yt)); scores[name].append(p)
print({n: len(v) for n, v in scores.items()}, "seeds loaded |", f"{len(yt):,} test rows, {int(yt.sum()):,} fraud")
A, B = scores["transformer"][0], scores["xgboost"][0]                 # the pair every row-level check uses
print("seed 0 at 0.6% alerts:", {n: round(evaluate_scores(yt, v[0], ct, ALERT)["recall_at_alert"], 3) for n, v in scores.items() if v})

## Stage 6a-2: is the lead real? Paired bootstrap on the same test rows

`bootstrap_diff` resamples whole cards (all of a card's test rows go in or out together) 1,000 times and recomputes metric(transformer) minus metric(XGBoost) each time. Card resampling, not row resampling: a card's transactions are correlated, and a row bootstrap would make the interval look narrower than it is. A 95% interval that excludes zero is the "statistically meaningful" the guide asks for; the seed-pair table shows the same difference across the three seed pairings as a second, cheaper view of the noise.

In [ ]:
t0 = time.time()
boot = bootstrap_diff(yt, A, B, ct, alert_rate=ALERT, n_boot=1000, seed=0, unit="card")
print(f"bootstrap: {boot['pr_auc']['n_boot']} card resamples in {time.time() - t0:.0f} s")
display(pd.DataFrame(boot).T.rename_axis("transformer minus xgboost")[["diff", "ci_low", "ci_high", "n_boot"]].style.format({"diff": "{:+.4f}", "ci_low": "{:+.4f}", "ci_high": "{:+.4f}"}))
pairs = []
for k in range(min(len(scores["transformer"]), len(scores["xgboost"]))):
    ma, mb = evaluate_scores(yt, scores["transformer"][k], ct, ALERT), evaluate_scores(yt, scores["xgboost"][k], ct, ALERT)
    pairs.append({"seed": k, **{f"{m} diff": ma[m] - mb[m] for m in ("pr_auc", "recall_at_alert", "fp_ratio")}})
pairs = pd.DataFrame(pairs); display(pairs.style.format("{:+.4f}", subset=[c for c in pairs if c != "seed"]).hide(axis="index"))
for m in ("pr_auc", "recall_at_alert", "fp_ratio"):
    b = boot[m]; verdict = "excludes zero" if (b["ci_low"] > 0 or b["ci_high"] < 0) else "includes zero"
    print(f"{m:16s} {b['diff']:+.4f}  95% CI [{b['ci_low']:+.4f}, {b['ci_high']:+.4f}]  {verdict}")

## Stage 6a-3: stability over time and input drift

Two separate questions. First the output side: cut the test period by month, keep the cutoff fixed, and read alert rate, recall and FP ratio per month; a model whose alerts pile up in one month or whose recall decays is a different operational risk from one that is merely a bit worse. Then the input side: PSI of every token field between the train period and the test period, the drift check Wealthsimple runs on the data before it reaches the model. PSI below 0.1 is usually read as stable, 0.1 to 0.25 as a moderate shift, above 0.25 as a major shift.

In [ ]:
import matplotlib.pyplot as plt
ts_t = test["ts"].to_numpy()
monthly = {n: monthly_metrics(yt, scores[n][0], ct, ts_t, ALERT) for n in ("transformer", "xgboost") if scores[n]}
m = monthly["transformer"][["month", "n", "n_fraud", "fraud_rate"]].copy()
for n, t in monthly.items():
    for c in ("alert_rate", "recall_at_cutoff", "fp_ratio", "pr_auc"): m[f"{c} {n[:3]}"] = t[c].to_numpy()
display(m.style.format({c: "{:.3f}" for c in m if c not in ("month", "n", "n_fraud")}).format({"n": "{:,}", "fraud_rate": "{:.2%}"}).hide(axis="index"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for n, t in monthly.items():
    ax[0].plot(t["month"], t["recall_at_cutoff"], marker="o", label=n); ax[1].plot(t["month"], t["alert_rate"] * 100, marker="o", label=n)
ax[0].set_title("recall at the fixed cutoff, by month"); ax[0].set_ylim(0, 1.02); ax[0].legend()
ax[1].set_title("alert rate by month (% of rows), cutoff fixed on the whole test set"); ax[1].axhline(ALERT * 100, ls="--", c="gray")
for a in ax: a.tick_params(axis="x", rotation=45)
plt.tight_layout(); fig.savefig(f"{RES}/06a_monthly.png", dpi=120); plt.show()

psi_tt = psi_by_field(ids_all[split == "train"], ids_all[te], FIELDS).rename(columns={"psi": "psi train vs test"})
psi_tv = psi_by_field(ids_all[split == "train"], ids_all[split == "val"], FIELDS)["psi"].to_numpy()
psi_tt["psi train vs val"] = psi_tv
psi_tt["reading"] = np.select([psi_tt["psi train vs test"] < 0.1, psi_tt["psi train vs test"] < 0.25], ["stable", "moderate shift"], "major shift")
display(psi_tt.style.format({"psi train vs test": "{:.4f}", "psi train vs val": "{:.4f}"}).hide(axis="index"))

## Stage 6a-4: subgroups

Recall at the fixed cutoff per group, both models side by side; groups with fewer than 20 test frauds are dropped because a recall over 5 frauds is noise. Three cuts: transaction category; how much history the card had at that moment (rows before it, over the whole parquet, the same quantity the transformer's window and XGBoost's rolling features see); and the cardholder attributes from `cards.parquet`, joined here for the first time and never used as a feature. Young cards ("0" and "1-4") are the risk the guide names: a sequence model with nothing to read.

In [ ]:
feats = build_features(df)                                                  # step-5 features, for hist_n and the error tables
pos_t = history_length(card)[te]
groups = {"category": test["category"].to_numpy(), "history rows before": history_bucket(pos_t)}
ct_attr = test[["card_id"]].merge(cards, on="card_id", how="left")
for c in ("gender", "age_bucket", "city_pop_bucket"):
    groups[c] = ct_attr[c].fillna("unknown").to_numpy()

def side_by_side(g, min_fraud=20):
    out = None
    for n in ("transformer", "xgboost"):
        s = subgroup_metrics(yt, scores[n][0], ct, g, ALERT, min_fraud=min_fraud)[["group", "n", "n_fraud", "recall_at_cutoff", "alert_rate", "pr_auc"]]
        s = s.rename(columns={"recall_at_cutoff": f"recall {n[:3]}", "alert_rate": f"alert rate {n[:3]}", "pr_auc": f"pr_auc {n[:3]}"})
        out = s if out is None else out.merge(s, on=["group", "n", "n_fraud"], how="outer")
    if out is not None and len(out) and "recall tra" in out and "recall xgb" in out:
        out["recall gap (tra - xgb)"] = out["recall tra"] - out["recall xgb"]
    return out

sub = {}
for name, g in groups.items():
    sub[name] = side_by_side(g)
    print(f"\n== by {name} ==  ({len(sub[name]) if sub[name] is not None else 0} groups with >= 20 test frauds)")
    if sub[name] is not None and len(sub[name]):
        order = ["0", "1-4", "5-11", "12-31", "32+"] if name == "history rows before" else None
        t = sub[name].set_index("group").loc[[o for o in order if o in sub[name]["group"].values]].reset_index() if order else sub[name]
        display(t.style.format({c: "{:.3f}" for c in t if c not in ("group", "n", "n_fraud")}).format({"n": "{:,}"}).hide(axis="index"))

## Stage 6a-5: error analysis, 20 + 20

Frauds that one model flags at its cutoff and the other does not, the 20 with the largest score gap each way. Each row shows the transaction, the step-5 history features that describe its context (how many earlier rows the card had, how far the amount is from the card's own mean, the gap to the previous transaction) and both scores; three of each side are decoded token by token, the card's previous 12 transactions and then the one being judged, so the pattern the model saw is visible. The medians table at the end is the summary: what the transformer-only catches look like versus the XGBoost-only catches versus all test fraud.

In [ ]:
only_tf, only_xgb = disagreements(yt, A, B, ALERT, k=20)
cols = ["ts", "category", "merchant", "amt", "dist_km"]
fcols = ["hist_n", "amt_ratio_mean", "amt_z", "gap_prev_s", "n_24h", "hist_cat_share", "first_at_merchant"]
ft = feats[te].reset_index(drop=True)
def show(idx, title):
    print(f"\n== {title}: {len(idx)} rows ==")
    if not len(idx): return
    t = pd.concat([test.loc[idx, cols].reset_index(drop=True), ft.loc[idx, fcols].reset_index(drop=True)], axis=1)
    t.insert(0, "test_row", idx); t["score tra"] = A[idx]; t["score xgb"] = B[idx]
    display(t.style.format({"amt": "{:.2f}", "dist_km": "{:.1f}", "amt_ratio_mean": "{:.2f}", "amt_z": "{:.2f}", "gap_prev_s": "{:,.0f}", "hist_cat_share": "{:.2f}", "score tra": "{:.4f}", "score xgb": "{:.4f}"}).hide(axis="index"))
show(only_tf, "fraud flagged by the transformer only"); show(only_xgb, "fraud flagged by XGBoost only")

test_pos = np.flatnonzero(te)                                               # parquet row of each test row
def decode_history(i, H=12):
    r = test_pos[i]; first = r - min(pos_t[i], H)
    lines = []
    for q in range(first, r + 1):
        toks = tok.decode(ids_all[q]); mark = "  <- judged" if q == r else ""
        lines.append(f"  {df['ts'].iloc[q]:%m-%d %H:%M}  fraud={df['is_fraud'].iloc[q]}  {' '.join(toks[1:])}{mark}")
    return "\n".join(lines)
for title, idx in (("transformer only", only_tf[:3]), ("xgboost only", only_xgb[:3])):
    for i in idx:
        print(f"\n-- {title}: test row {i}, score tra {A[i]:.4f} xgb {B[i]:.4f} --\n{decode_history(i)}")

all_fraud = np.flatnonzero(yt == 1)
med = pd.DataFrame({"transformer only": ft.loc[only_tf, fcols + ['amt']].median() if len(only_tf) else np.nan,
                    "xgboost only": ft.loc[only_xgb, fcols + ['amt']].median() if len(only_xgb) else np.nan,
                    "all test fraud": ft.loc[all_fraud, fcols + ['amt']].median(),
                    "all test rows": ft[fcols + ['amt']].median()}).T
med.insert(0, "rows", [len(only_tf), len(only_xgb), len(all_fraud), len(ft)])
print("\nmedians of the context features in each set:"); display(med.style.format("{:.2f}").format({"rows": "{:,}", "gap_prev_s": "{:,.0f}", "hist_n": "{:.0f}"}))

## Stage 6a-6: the ceiling, or the Sparkov caveat in numbers

Sparkov generates fraud from rules, so a model can learn the rules. Two numbers make the caveat concrete: how much of the test fraud each model scores above 0.99, and how far the typical fraud amount sits from the typical normal amount. When most fraud is trivially separable, the report has to say that the absolute levels carry little information and that the comparison lives in the fixed-alert-rate gaps, the subgroup gaps and the ablation trends.

In [ ]:
ceiling = {}
for n, v in scores.items():
    if v:
        s = v[0]; c = global_cutoff(s, ALERT)
        ceiling[n] = {"fraud scored > 0.99": float((s[yt == 1] > 0.99).mean()), "cutoff at 0.6% alerts": float(c),
                      "normals above cutoff": float((s[yt == 0] >= c).mean())}
display(pd.DataFrame(ceiling).T.style.format({"fraud scored > 0.99": "{:.1%}", "cutoff at 0.6% alerts": "{:.4f}", "normals above cutoff": "{:.3%}"}))
amt = {"fraud median amt": float(test.loc[yt == 1, "amt"].median()), "normal median amt": float(test.loc[yt == 0, "amt"].median()),
       "fraud share with amt > 200": float((test.loc[yt == 1, "amt"] > 200).mean()), "normal share with amt > 200": float((test.loc[yt == 0, "amt"] > 200).mean())}
print({k: round(v, 3) for k, v in amt.items()})

## Stage 6a-7: save 06a_validate.json

In [ ]:
def records(d): return None if d is None else d.replace([np.inf, -np.inf], np.nan).to_dict(orient="records")
out = {"alert_rate": ALERT, "n_test": int(len(yt)), "n_test_fraud": int(yt.sum()), "pair": "transformer pretrained_H12 seed0 vs xgboost full seed0",
       "bootstrap": boot, "seed_pairs": records(pairs), "monthly": {n: records(t) for n, t in monthly.items()}, "psi": records(psi_tt),
       "subgroups": {n: records(t) for n, t in sub.items()},
       "disagreements": {"transformer_only": [int(i) for i in only_tf], "xgboost_only": [int(i) for i in only_xgb], "medians": med.to_dict(orient="index")},
       "ceiling": ceiling, "amount_separation": amt, "runtime": describe()}
with open(f"{RES}/06a_validate.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved 06a_validate.json:", list(out))

## What to read off the numbers

- The bootstrap interval is the one sentence the report needs about transformer versus XGBoost. If it excludes zero, the lead is real at this test size; if it includes zero, the honest statement is that the two models are tied on this data, and the cost comparison from step 5 decides.
- Monthly: alert rate per month is the operational number. A cutoff fixed on the whole period can fire far more in one month than another; a validator asks whether the alert budget would have been blown. Recall per month tells whether the model's edge holds in every month or comes from one.
- PSI: Sparkov has no real drift to find; the expected reading is "stable" on every field, with the test period's own calendar (hour, weekday) the only mover. On real data this table is where a model gets sent back before it is scored.
- Subgroups: the "0" and "1-4" history buckets are the ones to quote. WF's claim that long history is the biggest driver has a mirror image: a card with no history gives the sequence model nothing, and the gap to XGBoost there is the deployment risk.
- Error analysis: the medians table, not the individual rows, is the finding. Which context features separate the transformer-only catches from the XGBoost-only catches says what each model reads.
- Ceiling: if most fraud scores above 0.99 for every model, write that the levels are saturated and point the reader to the gaps and the ablations, as the guide says.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
if on_colab():
    def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
        cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
        return cells(a) == cells(b)
    for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
        repo = f"notebooks/{os.path.basename(f)}"
        if not os.path.exists(repo) or same_code(f, repo):
            shutil.copy(f, repo)
        else:
            print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
    !git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
    !git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
    !git log --oneline | head -1
else:
    print("laptop: results copied into results/; this notebook file already is the repo copy; commit from the terminal")
    !git status --short results notebooks

In [ ]:
if on_colab():
    drive.flush_and_unmount()